# Notebook 01 — Exploratory Data Analysis
**Fashion Product Images Dataset** — class distributions, image statistics, label co-occurrence.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from data.splits import load_and_prepare_metadata, build_label_matrix
from config import CFG

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load raw metadata


In [ ]:
CSV_PATH = CFG.styles_csv

if not Path(CSV_PATH).exists():
    print(f'[ERROR] {CSV_PATH} not found. Run: python download_data.py')
else:
    df_raw = pd.read_csv(CSV_PATH, on_bad_lines='skip')
    print(f'Shape: {df_raw.shape}')
    display(df_raw.head(5))

## 2. Missing values


In [ ]:
target_cols = CFG.target_columns
print('Missing values per column:')
print(df_raw[target_cols].isnull().sum())

fig, axes = plt.subplots(1, len(target_cols), figsize=(16, 3))
for ax, col in zip(axes, target_cols):
    miss = df_raw[col].isnull().sum()
    ax.bar(['Present', 'Missing'], [len(df_raw) - miss, miss],
           color=['#4C72B0', '#DD8452'])
    ax.set_title(col); ax.set_ylabel('Count')
plt.suptitle('Missing values per target column', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/eda_missing.png', dpi=120)
plt.show()

## 3. Class distributions


In [ ]:
df, encoder, num_classes, class_names = load_and_prepare_metadata(
    CSV_PATH, target_columns=target_cols
)
print(f'After cleaning: {len(df)} rows, num_classes={num_classes}')

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, col in zip(axes, target_cols):
    vc = df[col].value_counts().head(20)
    vc.plot(kind='barh', ax=ax, color='#4C72B0', alpha=0.85)
    ax.set_title(f'{col}  (top 20 of {df[col].nunique()})',
                 fontweight='bold')
    ax.set_xlabel('Count')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_width()):,}',
                    (p.get_width() + 50, p.get_y() + p.get_height()/2),
                    va='center', fontsize=7)
plt.suptitle('Class Distributions per Target Column', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/eda_class_dist.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Class imbalance statistics


In [ ]:
from training.losses import compute_pos_weight
Y = build_label_matrix(df, encoder)
pos_counts = Y.sum(axis=0)

print('Label matrix shape:', Y.shape)
print(f'Average positives per sample: {Y.sum(axis=1).mean():.2f}')
print(f'Most common class: {class_names[pos_counts.argmax()]}  ({int(pos_counts.max()):,} samples)')
print(f'Rarest  class:     {class_names[pos_counts.argmin()]}  ({int(pos_counts.min()):,} samples)')

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(pos_counts, bins=40, color='#55A868', edgecolor='white')
ax.set_xlabel('Positive sample count per class')
ax.set_ylabel('Number of classes')
ax.set_title('Class frequency distribution (label imbalance)', fontweight='bold')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('../results/eda_imbalance.png', dpi=120)
plt.show()

## 5. Label co-occurrence heatmap


In [ ]:
master_cats = df['masterCategory'].value_counts().index[:10].tolist()
article_types = df['articleType'].value_counts().index[:10].tolist()

comat = pd.crosstab(
    df[df['masterCategory'].isin(master_cats)]['masterCategory'],
    df[df['articleType'].isin(article_types)]['articleType'],
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(comat, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.3,
            ax=ax, cbar_kws={'label': 'Sample count'})
ax.set_title('masterCategory × articleType co-occurrence (top 10 each)',
             fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../results/eda_cooccurrence.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Sample images grid


In [ ]:
from PIL import Image

IMAGE_DIR = Path(CFG.image_dir)
sample_df = df.sample(16, random_state=42)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    img_path = IMAGE_DIR / f"{row['id']}.jpg"
    if img_path.exists():
        ax.imshow(Image.open(img_path))
    else:
        ax.set_facecolor('#ddd')
    ax.set_title(
        f"{row['articleType']}\n{row['masterCategory']}",
        fontsize=7
    )
    ax.axis('off')

plt.suptitle('Sample Product Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/eda_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Dataset summary
All EDA figures saved to `results/`.


In [ ]:
print('=== Dataset Summary ===')
print(f'Total samples:      {len(df):,}')
print(f'num_classes:        {num_classes}')
for col in target_cols:
    print(f'  {col}: {df[col].nunique()} unique values')
print(f'Label matrix:       {Y.shape}')
print(f'Avg labels/sample:  {Y.sum(axis=1).mean():.2f}')
print(f'\nImage dir:          {IMAGE_DIR}')
n_imgs = len(list(IMAGE_DIR.glob("*.jpg"))) if IMAGE_DIR.exists() else 0
print(f'Images on disk:     {n_imgs:,}')